In [142]:
import pandas as pd
import numpy as np

# google_trends = pd.read_csv("data/gold_google_trends_daily.csv")
data = pd.read_csv("files/processed_data.csv")
match_context = pd.read_csv("data/gold_match_context.csv")
# match_goals = pd.read_csv("data/gold_match_goals.csv")
match_tickets = pd.read_csv("data/gold_match_tickets.csv")
matches = pd.read_csv("data/gold_match.csv")
match_articles = pd.read_csv("data/gold_belga_press_articles.csv", on_bad_lines="skip")

In [143]:
article_count = (match_articles
                 .groupby("match_id")["match_id"]
                 .value_counts()
                 .to_frame()
                 .reset_index()
                 .rename(columns={"count": "article_count"}))

In [144]:
tickets_sold_goals = match_tickets[["tickets_sold_total", "match_id", "seasonpass_holders"]]
matchdays = matches[["matchday", "match_id"]]

data = pd.merge(data, tickets_sold_goals, on="match_id")
data = pd.merge(data, matchdays, on="match_id")
data["away_team_code"] = data["away_team_code"].astype("category")


In [145]:
data["date"] = pd.to_datetime(data["date"])
data["month"] = data['date'].dt.month
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
data = data.drop(["date", "month"], axis=1)

In [146]:
data["kickoff_time_local"] = pd.to_datetime(data["kickoff_time_local"], format='%H:%M:%S')
data["kickoff_time_local"] = pd.to_datetime(data["kickoff_time_local"], format='%H:%M:%S').dt.hour
data["is_18_hours"] = data["kickoff_time_local"] == 18
data = data.drop("kickoff_time_local", axis=1)

In [147]:
data["is_sunday"] = data["weekday"] == 6
data["is_fri_hol"] = data["weekday"].isin([4, 5, 6])

In [148]:
train_data = data[:-10]
test_data = data.tail(10)

In [149]:
avg_tickets_scanned = train_data.groupby("away_team_code")["tickets_scanned"].mean().to_frame().rename(columns={"tickets_scanned": "avg_tickets_scanned"})
match_context_part = match_context[["academic_week", "has_promotion", "match_id"]]

In [150]:
train_data = pd.merge(train_data, match_context_part, on="match_id")
train_data = pd.merge(train_data, article_count, on="match_id")
train_data = pd.merge(train_data, avg_tickets_scanned, on="away_team_code")

test_data = pd.merge(test_data, match_context_part, on="match_id")
test_data = pd.merge(test_data, article_count, on="match_id")
test_data = pd.merge(test_data, avg_tickets_scanned, on="away_team_code")

In [151]:
train_data = train_data.drop("match_id", axis=1)
test_data = test_data.drop("match_id", axis=1)

In [152]:
test_data
# train_data

,away_team_code,last_result_vs_opponent,tickets_scanned,weekday,ohl_interest,tickets_sold_total,seasonpass_holders,matchday,month_cos,is_18_hours,is_sunday,is_fri_hol,academic_week,has_promotion,article_count,avg_tickets_scanned
0,AND,0,6977.0,4,68.52,8111,4235,9.0,-1.836970e-16,False,False,True,4,True,109,10778.666667
1,CLU,-1,7992.0,5,100.00,8549,4235,11.0,5.000000e-01,True,False,True,7,True,72,9490.333333
2,GNT,0,6373.0,6,66.67,7347,4235,13.0,8.660254e-01,False,True,True,9,True,195,6812.500000
3,STV,1,5661.0,6,47.22,7078,4235,15.0,8.660254e-01,False,True,True,12,True,11,6555.500000
4,ZWA,-2,4494.0,6,39.81,6278,4235,17.0,1.000000e+00,False,True,True,14,True,30,7785.000000
5,CER,1,5812.0,6,34.16,7473,4235,19.0,1.000000e+00,False,True,True,16,True,38,6095.666667
6,STG,-5,5322.0,5,47.30,6720,4235,22.0,8.660254e-01,False,False,True,21,True,6,7474.333333
7,KVM,0,5971.0,6,40.00,7355,4235,23.0,5.000000e-01,False,True,True,22,True,39,7662.800000
8,DEN,1,5137.0,5,40.00,7118,4235,25.0,5.000000e-01,False,False,True,24,True,117,5017.000000
9,WES,-2,6062.0,5,53.00,7526,4235,28.0,6.123234e-17,False,False,True,27,True,44,6566.800000


In [153]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error


X_train = train_data.drop("tickets_scanned", axis=1)
y_train = train_data["tickets_scanned"]

X_test = test_data.drop("tickets_scanned", axis=1)
y_test = test_data["tickets_scanned"]

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    enable_categorical=True
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)

MAE: 1346.03466796875
